In [1]:
# Imports
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from ydata_profiling import ProfileReport

In [2]:
# DPCFam Standard Mcs Properties
df1 = pd.read_csv("/u/mdmc/enyanduk/internship_areasciencepark/Dataframes/DPCFam/dpcfam_mcs_props.csv")
# Drop size_pfam column
df1 = df1.drop(columns=["size_pfam"])
df1.head()

,mcid,size_uniref50,avg_len,std_avg_len,lc_percent,cc_percent,dis_percent,tm,pfam_da,da_percent,avg_ov_percent,overlap_label
0,MC1,17931,185.68,28.77,4.72,0.00,18.44,0.01,PF13614,44.23,80.82,equivalent
1,MC4,617,59.91,6.07,4.99,0.00,1.87,1.26,PF03600,62.84,7.54,shifted
2,MC15,139,81.21,5.05,4.96,0.18,13.81,0.03,UNKNOWN,0.00,0.00,NONE
3,MC19,120,71.57,7.70,8.85,2.25,4.86,1.69,PF11915,94.07,13.66,shifted
4,MC21,937,91.20,7.70,2.65,0.00,24.38,0.00,PF01012,98.90,34.52,shifted


In [3]:
# DPCFamB Mcs Properties
df2 = pd.read_csv("/u/mdmc/enyanduk/internship_areasciencepark/Dataframes/DPCFam/DPCFamB/dpcfamB_mcs_props.csv")
df2.head()

,mcid,size_uniref50,avg_len,std_avg_len,lc_percent,cc_percent,dis_percent,tm,pfam_da,da_percent,avg_ov_percent,overlap_label
0,MC3,28,215.14,25.44,5.08,0.38,30.15,0.0,PF14631,8571.43,24.57,shifted
1,MC5,26,253.19,23.11,4.88,0.00,7.51,0.0,PF12937,6800.00,16.89,extended
2,MC7,32,65.53,3.30,3.39,0.00,0.10,0.0,PF01243,7407.41,6.27,shifted
3,MC11,32,134.97,11.99,2.05,0.00,15.61,0.0,UNKNOWN,10000.00,0.00,shifted
4,MC17,26,243.69,25.93,3.44,0.00,31.88,0.0,PF08590,9000.00,5.31,shifted


In [4]:
# We concatenate the two dataframes
df = pd.concat([df1, df2], ignore_index=True).drop_duplicates().reset_index(drop=True)
df.head()

,mcid,size_uniref50,avg_len,std_avg_len,lc_percent,cc_percent,dis_percent,tm,pfam_da,da_percent,avg_ov_percent,overlap_label
0,MC1,17931,185.68,28.77,4.72,0.00,18.44,0.01,PF13614,44.23,80.82,equivalent
1,MC4,617,59.91,6.07,4.99,0.00,1.87,1.26,PF03600,62.84,7.54,shifted
2,MC15,139,81.21,5.05,4.96,0.18,13.81,0.03,UNKNOWN,0.00,0.00,NONE
3,MC19,120,71.57,7.70,8.85,2.25,4.86,1.69,PF11915,94.07,13.66,shifted
4,MC21,937,91.20,7.70,2.65,0.00,24.38,0.00,PF01012,98.90,34.52,shifted


In [5]:
# In mcid column,we have values like MCID where ID is an integer. We wanna sort the dataframe based on ID in ascending order
# We can extract the integer part from the mcid column and sort the dataframe based on that
df["mcid_int"] = df["mcid"].str.extract(r"MC(\d+)").astype(int)
df = df.sort_values(by="mcid_int").reset_index(drop=True)
# We can drop the mcid_int column as we don't need it anymore
df = df.drop(columns=["mcid_int"])
# We capitalize the first letter of the overlap_label column
df["overlap_label"] = df["overlap_label"].str.capitalize()
df.head()

,mcid,size_uniref50,avg_len,std_avg_len,lc_percent,cc_percent,dis_percent,tm,pfam_da,da_percent,avg_ov_percent,overlap_label
0,MC1,17931,185.68,28.77,4.72,0.00,18.44,0.01,PF13614,44.23,80.82,Equivalent
1,MC3,28,215.14,25.44,5.08,0.38,30.15,0.00,PF14631,8571.43,24.57,Shifted
2,MC4,617,59.91,6.07,4.99,0.00,1.87,1.26,PF03600,62.84,7.54,Shifted
3,MC5,26,253.19,23.11,4.88,0.00,7.51,0.00,PF12937,6800.00,16.89,Extended
4,MC7,32,65.53,3.30,3.39,0.00,0.10,0.00,PF01243,7407.41,6.27,Shifted


In [6]:
# Infos
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 81384 entries, 0 to 81383
Data columns (total 12 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   mcid            81384 non-null  object 
 1   size_uniref50   81384 non-null  int64  
 2   avg_len         81384 non-null  float64
 3   std_avg_len     81384 non-null  float64
 4   lc_percent      81384 non-null  float64
 5   cc_percent      81384 non-null  float64
 6   dis_percent     81384 non-null  float64
 7   tm              81384 non-null  float64
 8   pfam_da         81384 non-null  object 
 9   da_percent      81384 non-null  float64
 10  avg_ov_percent  81384 non-null  float64
 11  overlap_label   81384 non-null  object 
dtypes: float64(8), int64(1), object(3)
memory usage: 7.5+ MB


In [7]:
# Fix pfam_da: insert "-" between concatenated Pfam/Clan IDs
# Handles both PF (7 chars: PF + 5 digits) and CL (6 chars: CL + 4 digits)
df["pfam_da"] = df["pfam_da"].fillna("UNKNOWN").str.replace(
    r'(PF\d{5}|CL\d{4})(?=PF|CL)', r'\1-', regex=True
)

# Verify: show rows that have separated IDs
sample = df[df["pfam_da"].str.contains("-", na=False)].head(10)
print(f"Rows with '-' in pfam_da: {df['pfam_da'].str.contains('-', na=False).sum()}")
sample[["mcid", "pfam_da"]]

Rows with '-' in pfam_da: 3162


,mcid,pfam_da
57,MC121,PF03783-PF13432
103,MC223,PF01424-PF12752
165,MC377,PF00307-PF00307
189,MC440,PF08305-PF08305
194,MC449,PF11707-PF16201
265,MC619,PF05008-PF12352
331,MC774,PF04239-PF07870
359,MC836,PF03682-PF00583
363,MC840,PF02550-PF13336
373,MC865,PF01852-PF01852


In [8]:
# Head
df.head()

,mcid,size_uniref50,avg_len,std_avg_len,lc_percent,cc_percent,dis_percent,tm,pfam_da,da_percent,avg_ov_percent,overlap_label
0,MC1,17931,185.68,28.77,4.72,0.00,18.44,0.01,PF13614,44.23,80.82,Equivalent
1,MC3,28,215.14,25.44,5.08,0.38,30.15,0.00,PF14631,8571.43,24.57,Shifted
2,MC4,617,59.91,6.07,4.99,0.00,1.87,1.26,PF03600,62.84,7.54,Shifted
3,MC5,26,253.19,23.11,4.88,0.00,7.51,0.00,PF12937,6800.00,16.89,Extended
4,MC7,32,65.53,3.30,3.39,0.00,0.10,0.00,PF01243,7407.41,6.27,Shifted


In [9]:
# Save the dataframe to csv files
df.to_csv("/u/mdmc/enyanduk/internship_areasciencepark/Dataframes/DPCFam/Merged/dpcfam_all_mcs_props.csv", index=False)